# Computer lab 3: duality

**Mathematical Optimization, VU Amsterdam, 2026–27.**

Derive each dual before implementing it. Keep track of the objective sense,
constraint orientation and variable domains. Check primal and dual feasibility
as well as their objective values, and explain what the multipliers mean.

## Setup

Run the next two cells before starting. Colab installs the required packages,
checks **HiGHS** for the LPs in Exercise 1 and prepares **Ipopt** for the quadratic
models in Exercise 2. Expand the setup code for inspection. If asked to restart
the session, restart and run Setup again. See the
[course setup instructions](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local use and troubleshooting.

In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import types
from urllib.request import urlopen

SETUP_COMMIT = '7a6ab1af4c59d39df2045116edba5d05058af6bd'
SETUP_SHA256 = "8886bca2ede3e8060891858a4cbc043f861ce4f6e0f5d3c45ad3782dba7dcd28"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    if SETUP_COMMIT is None:
        raise RuntimeError("This draft needs its published course setup pin before use on Colab.")
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    with urlopen(setup_location, timeout=30) as response:
        setup_source = response.read()
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=3)

In [ ]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

lp_solver, nlp_solver = lab.lp_solver, lab.nlp_solver
dual_display = lab.helpers["tutorial3"]
add_dual_variables = dual_display.add_dual_variables
display_dual_variables = dual_display.display_dual_variables
format_number = lab_setup.format_number

Check termination before loading a solution, then evaluate with `pyo.value(...)`.
Pass the solver explicitly. Ipopt checks local optimality conditions; convexity
and the primal–dual certificates below justify global conclusions here.
Formatting changes displayed text only; calculations retain full precision.

In [ ]:
def solve_and_load(model, solver):
    result = solver.solve(model, load_solutions=False)
    pyo.assert_optimal_termination(result)
    model.solutions.load_from(result)
    return result

Useful references are the MO-book's [dual of the BIM production problem](https://mobook.github.io/MO-book/notebooks/02/06-bim-dual.html)
and the [Pyomo suffix documentation](https://pyomo.readthedocs.io/en/6.10.1/explanation/modeling/math_programming/suffixes.html).
The BIM example is a maximization primal; the two primals below are minimization
problems, so check their signs explicitly.

## Imported row duals and Lagrangian multipliers

For a **minimization primal**, the solvers used here report a raw RHS dual $d_i$
for each constraint row:

| Row orientation | Sign of raw RHS dual |
|:---|:---|
| $a_i^\top x\ge b_i$ | $d_i\ge0$ |
| $a_i^\top x\le b_i$ | $d_i\le0$ |
| $a_i^\top x=b_i$ | $d_i$ unrestricted |

Where the optimal-value function is differentiable, $d_i$ is its sensitivity to
increasing the RHS $b_i$ of that row. Here "RHS" means the row's bound $b_i$,
wherever Pyomo prints it. Negating a row changes the reported dual sign.
Exercise 1 uses LP dual variables $y_i$ in this RHS convention.

In Exercise 2 we write every inequality as $g_i(x)\le0$ and use
$L(x,\lambda)=f(x)+\sum_i\lambda_i g_i(x)$ with $\lambda_i\ge0$.
For these upper-bound rows, **$\lambda_i=-d_i$**. Keep the two sign conventions
separate when comparing imported values with a hand-derived dual.

Attach `model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)` **before solving**.
After a successful solve, read `model.dual[model.row[i]]` for an indexed row.
The imported course function `add_dual_variables(model)` also attaches this suffix.

## Exercise 1: a linear primal and its dual

Consider the LP

$$
\begin{aligned}
\min\quad &x_1-2x_2+4x_3\\
\text{s.t.}\quad &-x_1+3x_2=3 &&(1),\\
&2x_1-x_2+3x_3\ge-5 &&(2),\\
&x_3\le7 &&(3),\\
&x_1+x_2-x_3\le11 &&(4),\\
&x_1,x_2,x_3\in\mathbb{R}.
\end{aligned}
$$

**(a)** Implement and solve the primal with HiGHS. Keep all three variables free
and attach a dual suffix before solving.

**(b)** Derive the dual by hand. Explain the sign of each dual variable and why
the dual constraints are equalities. Implement and solve the dual with HiGHS.

**(c)** Read the imported row duals directly from the primal suffix and compare
them with your explicit dual solution. You can also use
`display_dual_variables(model)` to display the rows and their raw duals.
Does strong duality hold, and could you have predicted it? Interpret the signs
of the nonzero RHS duals in this example.

In [ ]:
linear_rhs = {1: 3, 2: -5, 3: 7, 4: 11}

In [ ]:
# Build the primal with real-valued variables and the original row orientations.
# Attach a dual suffix before solving with solve_and_load(model, lp_solver).
# Report the decisions and objective value.

In [ ]:
# Derive the dual and its variable domains from the original LP.
# Implement the sign restrictions as variable domains and solve with HiGHS.
# Report the four dual variables and objective value.

In [ ]:
# Read model.dual[model.row[i]] after the primal solve and compare with your dual.
# Call display_dual_variables(model) to see each coded row and its imported sign.
# Check the objective gap and explain strong duality and the nonzero dual signs.

## Exercise 2: a convex quadratic primal and its Lagrangian dual

Consider

$$
\begin{aligned}
\min\quad &f(x)=x_1^2+x_2^2+x_3^2\\
\text{s.t.}\quad &g_1(x)=2x_1+x_2-5\le0,\\
&g_2(x)=x_1+x_3-2\le0,\\
&g_3(x)=1-x_1\le0,\\
&g_4(x)=2-x_2\le0,\\
&g_5(x)=-x_3\le0,\qquad x\in\mathbb{R}^3.
\end{aligned}
$$

**(a)** Explain convexity, implement the primal and solve it with Ipopt. Keep the
five inequalities in the stated $g_i(x)\le0$ orientation. Import the raw row duals
and convert them to the nonnegative multipliers in $L=f+\sum_i\lambda_i g_i$.

**(b)** Derive the Lagrangian dual by minimizing $L$ over the unrestricted vector
$x$. Implement and solve the resulting dual with Ipopt. Completing the square
gives a compact expression for the dual function.

**(c)** Check strong duality and explain whether it was predictable. Verify
complementary slackness and identify the binding constraints. Can a binding
constraint have a zero multiplier in this example?

In [ ]:
# Matrix form of the five stated inequalities: g(x) = A @ x - b <= 0.
quadratic_A = np.array([
    [2, 1, 0],
    [1, 0, 1],
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, -1],
], dtype=float)
quadratic_b = np.array([5, 2, -1, -2, 0], dtype=float)

In [ ]:
# Build the quadratic primal using g_i(x)<=0 exactly as written.
# Keep all five rows explicit and attach a dual suffix before solving with Ipopt.
# Read each indexed row's raw dual and form lambda_i = -raw_dual_i.
# Report the decisions, objective, raw duals and converted multipliers.

In [ ]:
# Complete the square in x to derive ell(lambda), with lambda>=0.
# Build an unsolved model maximizing the concave quadratic dual function.
# Solve with Ipopt and compare its multipliers and objective with part (a).

In [ ]:
# Explain strong duality, using convexity and a suitable Slater argument.
# Check primal/dual feasibility, stationarity and complementary slackness.
# Identify the binding rows and determine whether each must have a positive multiplier.
# Retain raw numerical values when checking residuals and the objective gap.